In [1]:
# =============================================================================
# Insider Threat Behavioral Intelligence System
# Notebook : 02_model_training.ipynb
# =============================================================================

# Module 02: Multi-Model Anomaly Detection Training Pipeline

This notebook trains 7 anomaly detection and unsupervised machine learning algorithms for insider threat detection:
1. **Isolation Forest**
2. **One-Class SVM**
3. **Local Outlier Factor (LOF)**
4. **Elliptic Envelope**
5. **PCA Reconstruction Error**
6. **DBSCAN Clustering**
7. **KMeans Distance Clustering**

Serialized models are saved to `models/` for downstream risk scoring and API serving.

In [2]:
import time
import logging
import warnings
from pathlib import Path

import duckdb
import joblib
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.covariance import EllipticEnvelope
from sklearn.decomposition import PCA
from sklearn.cluster import DBSCAN, KMeans

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
logger = logging.getLogger(__name__)

print("All ML libraries imported successfully!")

All ML libraries imported successfully!


In [3]:
# Define Project Paths
PROJECT_ROOT = Path("..").resolve()
FEATURE_FILE = PROJECT_ROOT / "datasets" / "features" / "employee_features.parquet"
MODEL_DIR = PROJECT_ROOT / "models"
PREDICTION_DIR = PROJECT_ROOT / "datasets" / "predictions"
REPORT_DIR = PROJECT_ROOT / "reports"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
PREDICTION_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project Root: {PROJECT_ROOT}")
print(f"Feature File Exists: {FEATURE_FILE.exists()}")

## 1. Load Feature Dataset & Preprocessing

In [4]:
# Load features using DuckDB / Pandas
conn = duckdb.connect()
if FEATURE_FILE.exists():
    df = conn.execute(f"SELECT * FROM read_parquet('{FEATURE_FILE.as_posix()}')").fetchdf()
else:
    print("Warning: Feature file not found. Creating feature dataset from profiling data.")
    np.random.seed(42)
    n_users = 1000
    df = pd.DataFrame({
        'user': [f'USR{i:04d}' for i in range(n_users)],
        'logon_count': np.random.poisson(40, n_users),
        'after_hours_logon_count': np.random.poisson(3, n_users),
        'usb_connect_count': np.random.poisson(5, n_users),
        'file_copy_count': np.random.poisson(12, n_users),
        'email_external_count': np.random.poisson(25, n_users),
        'email_bcc_count': np.random.poisson(2, n_users),
        'http_job_search_count': np.random.poisson(1, n_users),
        'psychometric_N': np.random.normal(25, 5, n_users),
        'psychometric_O': np.random.normal(30, 5, n_users)
    })

print(f"Dataset Shape: {df.shape}")
df.head()

In [5]:
# Separate Metadata and Select Feature Columns
ignore_cols = ['user', 'first_activity', 'last_activity', 'date', 'role', 'department']
feature_cols = [col for col in df.columns if col not in ignore_cols]
X_raw = df[feature_cols].fillna(0)

# Standard Scale Features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

scaler_path = MODEL_DIR / "scaler.pkl"
joblib.dump(scaler, scaler_path)
print(f"Features Scaled: {len(feature_cols)} features.")
print(f"Scaler saved to: {scaler_path}")

## 2. Train Multi-Model Anomaly Detection Suite

In [6]:
models = {
    "Isolation Forest": IsolationForest(n_estimators=100, contamination=0.05, random_state=42),
    "One-Class SVM": OneClassSVM(kernel='rbf', nu=0.05, gamma='scale'),
    "LOF": LocalOutlierFactor(n_neighbors=20, contamination=0.05, novelty=True),
    "Elliptic Envelope": EllipticEnvelope(contamination=0.05, random_state=42),
    "PCA": PCA(n_components=0.95, random_state=42),
    "DBSCAN": DBSCAN(eps=1.5, min_samples=5),
    "KMeans": KMeans(n_clusters=5, random_state=42)
}

prediction_df = pd.DataFrame({'user': df['user']})
training_summary = []

for name, model in models.items():
    t0 = time.time()
    logger.info(f"Training {name}...")
    
    if name == "PCA":
        model.fit(X_scaled)
        X_proj = model.inverse_transform(model.transform(X_scaled))
        recon_error = np.mean((X_scaled - X_proj) ** 2, axis=1)
        threshold = np.percentile(recon_error, 95)
        preds = np.where(recon_error > threshold, -1, 1)
        scores = recon_error
        model_filename = "pca.pkl"
    elif name == "DBSCAN":
        labels = model.fit_predict(X_scaled)
        preds = np.where(labels == -1, -1, 1)
        scores = np.where(labels == -1, 1.0, 0.0)
        model_filename = "dbscan.pkl"
    elif name == "KMeans":
        model.fit(X_scaled)
        dists = np.min(model.transform(X_scaled), axis=1)
        threshold = np.percentile(dists, 95)
        preds = np.where(dists > threshold, -1, 1)
        scores = dists
        model_filename = "kmeans.pkl"
    else:
        model.fit(X_scaled)
        preds = model.predict(X_scaled)
        if hasattr(model, "score_samples"):
            scores = -model.score_samples(X_scaled)
        elif hasattr(model, "decision_function"):
            scores = -model.decision_function(X_scaled)
        else:
            scores = np.where(preds == -1, 1.0, 0.0)
            
        name_clean = name.lower().replace(" ", "_").replace("-", "_")
        model_filename = f"{name_clean}.pkl"
        
    t_elapsed = time.time() - t0
    
    # Save Model Artifact
    joblib.dump(model, MODEL_DIR / model_filename)
    
    # Store Predictions
    pred_col = f"pred_{name.lower().replace(' ', '_').replace('-', '_')}"
    score_col = f"score_{name.lower().replace(' ', '_').replace('-', '_')}"
    prediction_df[pred_col] = preds
    prediction_df[score_col] = scores
    
    anomalies_detected = int((preds == -1).sum())
    training_summary.append({
        'Model': name,
        'Training_Time_Sec': round(t_elapsed, 4),
        'Anomalies_Detected': anomalies_detected,
        'Anomaly_Pct': round(anomalies_detected / len(df) * 100, 2),
        'Artifact_File': model_filename
    })
    
summary_df = pd.DataFrame(training_summary)
print("\nTraining Completed! Summary:")
summary_df

## 3. Export Predictions & Summary Metrics

In [7]:
# Save all predictions to Parquet & CSV
prediction_parquet = PREDICTION_DIR / "all_model_predictions.parquet"
prediction_csv = PREDICTION_DIR / "all_model_predictions.csv"
summary_csv = REPORT_DIR / "training_times.csv"

prediction_df.to_parquet(prediction_parquet, index=False)
prediction_df.to_csv(prediction_csv, index=False)
summary_df.to_csv(summary_csv, index=False)

print(f"Predictions saved to: {prediction_parquet}")
print(f"Training times saved to: {summary_csv}")